In [ ]:
import pandas as pd
from google.colab import files

In [ ]:
# Helper function to consolidate active binary flags (e.g., 1, '1', etc.) into clean strings
def consolidate_active_flags(row, target_columns, mapping_dict=None):
    active_items = []
    for col in target_columns:
        if col in row.index:
            val = row[col]
            if pd.notnull(val) and str(val).strip() not in ['', '0', 'nan', 'NaN']:
                clean_name = mapping_dict.get(col, col) if mapping_dict else col
                active_items.append(clean_name)
    return ", ".join(active_items) if active_items else 'Not Specified'



In [ ]:
# 1. Load the raw dataset, skipping the redundant main title row
file_path = '/content/datos-de-laboratorio.csv'
df_raw = pd.read_csv(file_path, sep=';', header=1, encoding='latin1')

# Trim whitespace from raw column names to prevent matching errors
df_raw.columns = df_raw.columns.str.strip()


# 2. Define column groups and standardization mapping dictionaries
sample_cols = ['AGUA', 'EFLUENTE']
analysis_cols = ['BACTER.', 'IN SITU', 'BASICO', 'COMPLETO']
operator_cols = ['AySAM', 'O.G.C.', 'D.G.E.', 'D.G.I.', 'OTROS', 'P.T.L.C.', 'RECLAMO']

analysis_mapping = {
    'BACTER.': 'Bacteriological',
    'IN SITU': 'In Situ',
    'BASICO': 'Basic',
    'COMPLETO': 'Complete'
}

operator_mapping = {
    'AySAM': 'AySAM',
    'O.G.C.': 'O.G.C.',
    'D.G.E.': 'D.G.E.',
    'D.G.I.': 'D.G.I.',
    'OTROS': 'Other',
    'P.T.L.C.': 'P.T.L.C.',
    'RECLAMO': 'Complaint'
}


In [ ]:
# 3. Process and clean data row by row
clean_rows = []

for idx, row in df_raw.iterrows():
    # Base fields using positional iloc indexing to prevent KeyError
    date_val = str(row.iloc[0]).strip() if pd.notnull(row.iloc[0]) else ''
    location_val = str(row.iloc[1]).strip() if pd.notnull(row.iloc[1]) else ''

    # Department column (index 2)
    dept_val = row.iloc[2]
    dept_clean = str(dept_val).strip() if pd.notnull(dept_val) and str(dept_val).strip() not in ['nan', 'NaN', ''] else 'Not Specified'

    # Consolidate categorical flag groups
    sample_type = consolidate_active_flags(row, sample_cols)
    if sample_type == 'AGUA':
        sample_type = 'Water'
    elif sample_type == 'EFLUENTE':
        sample_type = 'Effluent'

    analysis_type = consolidate_active_flags(row, analysis_cols, analysis_mapping)
    operator = consolidate_active_flags(row, operator_cols, operator_mapping)

    # Observations field standardization (casing & text trimming)
    obs_raw = str(row.iloc[16]).strip() if len(row) > 16 and pd.notnull(row.iloc[16]) else ''
    if obs_raw.lower() in ['nan', '']:
        obs_clean = ''
    else:
        # Standardize 'factibilidad' -> 'Feasibility'
        if obs_raw.lower().startswith('factibilidad'):
            obs_clean = 'Feasibility' + obs_raw[12:]
        else:
            obs_clean = obs_raw[0].upper() + obs_raw[1:]

    clean_rows.append({
        'Date': date_val,
        'Extraction_Site': location_val,
        'Department': dept_clean,
        'Sample_Type': sample_type,
        'Analysis_Type': analysis_type,
        'Operator_Client': operator,
        'Notes': obs_clean
    })


In [ ]:
# 4. Create cleaned DataFrame and strip all remaining whitespace
df_clean = pd.DataFrame(clean_rows)

for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()



In [ ]:
# 5. Export processed dataset to native Excel (.xlsx) and regional CSV (.csv)
df_clean.to_excel('laboratory_data_cleaned.xlsx', index=False)
df_clean.to_csv('laboratory_data_cleaned.csv', sep=';', index=False, encoding='utf-8-sig')

print("ETL Pipeline completed successfully! Cleaned dataset ready.")

# Download clean Excel file automatically
files.download('laboratory_data_cleaned.xlsx')